# Day 5 — Lesson 110 Challenge: **LLM Trading Code Generator & Simulator** (Frontier + Open‑Source)

This notebook fulfills the final challenge: *a code generator that writes trading code to buy/sell equities in a simulated environment, based on a given API.* It is designed to run on **Google Colab with a T4 GPU** (or locally) and supports both **frontier** and **open‑source** model paths.

### What it does
- **Code generation** (Python): Use an LLM to write a `Strategy` class that trades via a simplified **Broker API**.
- **Backtesting**: Fetch historical OHLCV (via free `yfinance`), simulate fills (slippage/fees), and compute metrics (P&L, CAGR, Sharpe, Max Drawdown, Win rate).
- **UI**: Gradio app to (1) generate strategy code from a natural language spec and (2) backtest it.

> ⚠️ **Educational use only.** This simulator is for research/testing. **Do not** use any generated code for live trading. No financial advice.

## 0) Environment check

In [20]:
import os, sys, platform
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

print("Python:", sys.version)
print("Platform:", platform.platform())

try:
    import torch
    print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA version:", torch.version.cuda)
        
        # Test BitsAndBytes availability with better error handling
        try:
            import bitsandbytes as bnb
            print("✅ BitsAndBytes available - quantization enabled")
            quantization_available = True
        except Exception as bnb_e:
            print("⚠️ BitsAndBytes compilation failed (CUDA development headers missing)")
            print("   This is normal - will use CPU fallback for models")
            print("   Your CUDA runtime works fine, just missing development toolkit")
            quantization_available = False
    else:
        print("⚠️ CUDA not available - will use CPU mode")
        quantization_available = False
        
except Exception as e:
    print("Torch not installed yet", e)
    quantization_available = False

print(f"\n📋 System Summary:")
print(f"   CUDA Runtime: {'✅ Available' if torch.cuda.is_available() else '❌ Not available'}")
print(f"   Quantization: {'✅ Enabled' if quantization_available else '⚠️ CPU fallback'}")
print(f"   Status: {'🚀 Ready for GPU acceleration' if quantization_available else '💻 Ready for CPU inference'}")

Python: 3.11.13 | packaged by conda-forge | (main, Jun  4 2025, 14:48:23) [GCC 13.3.0]
Platform: Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39
Torch: 2.7.1 | CUDA available: True
GPU: NVIDIA GeForce RTX 3060
CUDA version: 12.9
⚠️ BitsAndBytes compilation failed (CUDA development headers missing)
   This is normal - will use CPU fallback for models
   Your CUDA runtime works fine, just missing development toolkit

📋 System Summary:
   CUDA Runtime: ✅ Available
   Quantization: ⚠️ CPU fallback
   Status: 💻 Ready for CPU inference


/tmp/tmprm0h3na8/main.c:1:10: fatal error: cuda.h: No such file or directory
    1 | #include "cuda.h"
      |          ^~~~~~~~
compilation terminated.


## 1) Installs

In [21]:
# !pip install -q -U transformers accelerate bitsandbytes sentencepiece gradio pandas numpy matplotlib yfinance python-dateutil
# !pip install -q -U openai==1.* anthropic==0.*

## 2) Imports & device

In [22]:
import os, re, io, json, math, time, tempfile, textwrap, types, contextlib
import datetime as dt
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf

import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Environment variable loading
from dotenv import load_dotenv

# Optional frontier clients

## 3) (Optional) Google Drive for datasets/results (Colab)

In [23]:
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    print("Drive mounted at /content/drive")
except Exception:
    print("Not running in Colab or Drive not available.")

Not running in Colab or Drive not available.


## 4) Auth & defaults

**Frontier models** (set env vars if you want to use them):
- `OPENAI_API_KEY` → models like `gpt-4o-mini`, `o4-mini`
- `ANTHROPIC_API_KEY` → models like `claude-3-5-sonnet-20240620` (or newer)

**Open‑source code LLM** (default fits T4 in 4‑bit):
- `Qwen/Qwen2.5-Coder-7B-Instruct`

In [24]:
# Load environment variables from .env file
load_dotenv(override=True)

# --- API keys ---
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", None)
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", None)

openai_client = None
if OPENAI_API_KEY and OpenAI is not None:
    try:
        openai_client = OpenAI(api_key=OPENAI_API_KEY)
        print("✅ OpenAI client ready")
    except Exception as e:
        print("⚠️ OpenAI client init failed:", e)

anthropic_client = None
if ANTHROPIC_API_KEY and anthropic is not None:
    try:
        anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        print("✅ Anthropic client ready")
    except Exception as e:
        print("⚠️ Anthropic client init failed:", e)

# --- Check if we can use advanced CUDA features ---
CUDA_ADVANCED = False
if torch.cuda.is_available():
    try:
        import bitsandbytes as bnb
        CUDA_ADVANCED = True
        print("✅ CUDA with quantization available")
    except Exception:
        print("⚠️ CUDA available but quantization disabled (missing CUDA toolkit)")

# --- Open‑source model defaults (CPU-friendly fallback) ---
if CUDA_ADVANCED:
    OS_DEFAULT = "Qwen/Qwen2.5-Coder-7B-Instruct"
else:
    OS_DEFAULT = "microsoft/Phi-3-mini-4k-instruct"  # Smaller, CPU-friendly
    print("🔄 Using CPU-optimized model as default")

OS_CHOICES = [
    "Qwen/Qwen2.5-Coder-7B-Instruct",
    "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct", 
    "HuggingFaceH4/zephyr-7b-beta",
    "microsoft/Phi-3-mini-4k-instruct",  # Best for CPU
    "microsoft/DialoGPT-medium",  # Fallback option
]

✅ OpenAI client ready
✅ Anthropic client ready
⚠️ CUDA available but quantization disabled (missing CUDA toolkit)
🔄 Using CPU-optimized model as default


/tmp/tmp7hmgl75r/main.c:1:10: fatal error: cuda.h: No such file or directory
    1 | #include "cuda.h"
      |          ^~~~~~~~
compilation terminated.


## 5) Lazy load open‑source LLM (4‑bit on GPU)

In [25]:
_tok = None
_mdl = None

def load_os_model(model_id: str):
    global _tok, _mdl
    if _mdl is not None and getattr(_mdl, "name_or_path", None) == model_id:
        return _tok, _mdl
    print(f"Loading open-source model: {model_id}")
    
    # Start with basic kwargs
    kwargs = {}
    
    # Try to use 4-bit quantization if CUDA is available and properly configured
    if DEVICE == "cuda":
        try:
            # Test if BitsAndBytesConfig can be imported and used
            from transformers import BitsAndBytesConfig
            import bitsandbytes as bnb  # This will fail if CUDA headers aren't available
            
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
            )
            kwargs["device_map"] = "auto"
            kwargs["torch_dtype"] = torch.bfloat16
            print("✅ Using 4-bit quantization with CUDA")
            
        except Exception as e:
            print(f"⚠️ 4-bit quantization failed ({e}), falling back to standard CUDA loading")
            try:
                kwargs["device_map"] = "auto"
                kwargs["torch_dtype"] = torch.float16
                print("✅ Using standard CUDA loading")
            except Exception as e2:
                print(f"⚠️ CUDA loading failed ({e2}), falling back to CPU")
                kwargs = {}
    
    # Load tokenizer
    _tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if _tok.pad_token is None:
        _tok.pad_token = _tok.eos_token
    
    # Load model with fallback logic
    try:
        _mdl = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
        print(f"✅ Model loaded successfully on {_mdl.device if hasattr(_mdl, 'device') else 'unknown device'}")
    except Exception as e:
        print(f"⚠️ Model loading with {kwargs} failed: {e}")
        print("🔄 Trying CPU fallback...")
        _mdl = AutoModelForCausalLM.from_pretrained(model_id)
        print("✅ Model loaded on CPU")
    
    return _tok, _mdl

## 6) **Broker API** (the interface your LLM strategy will use)

The LLM must **only** call these methods/attributes. No file/network I/O.

```python
class BrokerAPI:
    symbol: str
    def now(self) -> pd.Timestamp: ...
    def current_price(self) -> float: ...
    def history(self, n: int, field: str = "Close") -> List[float]:  # oldest→newest
    def position(self) -> float: ...
    def cash(self) -> float: ...
    def portfolio_value(self) -> float: ...
    def order_market_buy_units(self, qty: float) -> None: ...
    def order_market_sell_units(self, qty: float) -> None: ...
    def order_target_percent(self, target: float) -> None:  # 0..1, no leverage
```

In [26]:
@dataclass
class Trade:
    time: pd.Timestamp
    side: str
    price: float
    qty: float
    fee: float
    slippage: float
    cash_after: float
    position_after: float
    value_after: float

class BrokerAPI:
    def __init__(self, df: pd.DataFrame, symbol: str, initial_cash: float = 100_000.0, fee_bps: float = 1.0, slippage_bps: float = 1.0):
        assert {"Open","High","Low","Close","Volume"}.issubset(df.columns)
        self.df = df.copy()
        self.symbol = symbol
        self.initial_cash = float(initial_cash)
        self.fee_bps = float(fee_bps)
        self.slippage_bps = float(slippage_bps)
        self._i = 0
        self._position = 0.0  # units
        self._cash = float(initial_cash)
        self.trades: List[Trade] = []
        self.equity_curve: List[Tuple[pd.Timestamp, float]] = []

    def _fill_price(self) -> float:
        # Use Close with slippage
        p = float(self.df.iloc[self._i]["Close"])
        slip = p * (self.slippage_bps / 10_000.0)
        return p + slip, slip

    def now(self) -> pd.Timestamp:
        return pd.Timestamp(self.df.index[self._i])

    def current_price(self) -> float:
        return float(self.df.iloc[self._i]["Close"])

    def history(self, n: int, field: str = "Close") -> List[float]:
        k = max(0, self._i - n + 1)
        return list(map(float, self.df.iloc[k:self._i+1][field].values))

    def position(self) -> float:
        return float(self._position)

    def cash(self) -> float:
        return float(self._cash)

    def portfolio_value(self) -> float:
        return float(self._cash + self._position * self.current_price())

    def _order_units(self, qty: float):
        if qty == 0: 
            return
        price, slip = self._fill_price()
        notional = qty * price
        fee = abs(notional) * (self.fee_bps / 10_000.0)
        # No shorting or leverage in this simple engine
        if qty > 0:
            # buy
            max_affordable = math.floor(self._cash / (price + (self.fee_bps/10_000.0)*price))
            qty = min(qty, max_affordable)
            notional = qty * price
            fee = abs(notional) * (self.fee_bps / 10_000.0)
            self._cash -= (notional + fee)
            self._position += qty
            side = "BUY"
        else:
            # sell up to held
            qty = -min(-qty, self._position)
            notional = qty * price
            fee = abs(notional) * (self.fee_bps / 10_000.0)
            self._cash += (notional - fee)
            self._position -= qty
            side = "SELL"
        self.trades.append(Trade(time=self.now(), side=side, price=price, qty=float(qty),
                                 fee=float(fee), slippage=float(slip), cash_after=self._cash,
                                 position_after=self._position, value_after=self.portfolio_value()))

    # public methods expected by the strategy
    def order_market_buy_units(self, qty: float) -> None:
        self._order_units(float(qty))

    def order_market_sell_units(self, qty: float) -> None:
        self._order_units(-float(qty))

    def order_target_percent(self, target: float) -> None:
        target = max(0.0, min(1.0, float(target)))
        pv = self.portfolio_value()
        desired_value = pv * target
        current_value = self._position * self.current_price()
        delta_value = desired_value - current_value
        if abs(delta_value) < 1e-9:
            return
        units = math.floor(abs(delta_value) / self.current_price())
        if units <= 0:
            return
        if delta_value > 0:
            self._order_units(units)
        else:
            self._order_units(-units)

## 7) Data loader (free **yfinance**)

- Fetches OHLCV for a single symbol and date range.
- Adjusts for splits/dividends (`auto_adjust=True`).

In [27]:
def load_ohlcv(symbol: str, start: str, end: str, interval: str = "1d") -> pd.DataFrame:
    df = yf.download(symbol, start=start, end=end, interval=interval, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        # if user passed multiple symbols by mistake
        df = df.droplevel(0, axis=1)
    df = df.rename(columns=str.title)
    df = df[["Open","High","Low","Close","Volume"]].dropna()
    df.index = pd.DatetimeIndex(df.index)
    if df.empty:
        raise ValueError("No data returned. Check symbol/dates/interval.")
    return df

## 8) Metrics

In [28]:
def compute_metrics(equity: pd.Series, rf: float = 0.0) -> Dict[str, float]:
    # equity indexed by datetime
    rets = equity.pct_change().dropna()
    if rets.empty:
        return {"total_return": 0.0, "cagr": 0.0, "sharpe": 0.0, "max_drawdown": 0.0, "win_rate": 0.0}
    total_return = (equity.iloc[-1] / equity.iloc[0]) - 1.0
    # CAGR
    days = (equity.index[-1] - equity.index[0]).days
    years = max(1e-9, days/365.25)
    cagr = (equity.iloc[-1]/equity.iloc[0])**(1/years) - 1.0 if years > 0 else 0.0
    # Sharpe (daily → annualized, 252 trading days)
    excess = rets - (rf/252.0)
    sharpe = (excess.mean() / (excess.std()+1e-12)) * np.sqrt(252.0)
    # Max drawdown
    roll_max = equity.cummax()
    dd = equity/roll_max - 1.0
    max_dd = float(dd.min())
    # Win rate
    win_rate = float((rets > 0).mean())
    return {
        "total_return": float(total_return),
        "cagr": float(cagr),
        "sharpe": float(sharpe),
        "max_drawdown": float(max_dd),
        "win_rate": win_rate
    }

## 9) Backtest engine

In [29]:
def backtest(df: pd.DataFrame, symbol: str, strategy_obj, initial_cash: float = 100_000.0, fee_bps: float = 1.0, slippage_bps: float = 1.0):
    api = BrokerAPI(df=df, symbol=symbol, initial_cash=initial_cash, fee_bps=fee_bps, slippage_bps=slippage_bps)
    equity = []
    for i in range(len(df)):
        api._i = i
        # Update equity at bar open/close (here using close)
        equity.append((api.now(), api.portfolio_value()))
        # Strategy callback
        try:
            strategy_obj.on_bar(api)
        except Exception as e:
            raise RuntimeError(f"Strategy error on {api.now()}: {e}")
    equity = pd.Series([v for _, v in equity], index=[t for t, _ in equity], name="equity")
    trades_df = pd.DataFrame([t.__dict__ for t in api.trades])
    metrics = compute_metrics(equity)
    return {"equity": equity, "trades": trades_df, "metrics": metrics, "final_position": api.position(), "final_cash": api.cash()}

## 10) Prompting for **strategy generation**

The LLM must output **only** a Python code block defining:

```python
class Strategy:
    def __init__(self): ...
    def on_bar(self, api): ...
```

and must use **only** the `api` described earlier.

In [30]:
SYSTEM = (
        "You write clean, production-ready Python strategies for a simple backtesting engine. "
        "Return only code, no explanations, no markdown fences. "
        "Define exactly one class named Strategy with methods __init__(self) and on_bar(self, api). "
        "CRITICAL: The 'api' parameter is only available in on_bar method, NOT in __init__. "
        "Use ONLY these api methods (no parameters except where shown): "
        "api.now(), api.current_price(), api.history(n, field='Close'), "
        "api.position(), api.cash(), api.portfolio_value(), "
        "api.order_market_buy_units(qty), api.order_market_sell_units(qty), api.order_target_percent(target). "
        "The api.current_price(), api.position(), api.cash(), api.portfolio_value() methods take NO parameters. "
        "Do NOT store api references in __init__. Only use api in on_bar method. "
        "Do NOT read/write files or network. Avoid external dependencies; use stdlib only (e.g., math, statistics). "
        "Be robust to missing warmup by checking len(api.history(...)). "
        "Keep logic deterministic and simple."
    )

TEMPLATE_USER = """Write a trading strategy for symbol-level daily bars using the Broker API. 

Requirements:
- Define class Strategy with __init__(self) and on_bar(self, api) methods
- The api parameter is ONLY available in on_bar method, not in __init__
- Use only the methods listed in the API (no extra parameters)
- Use daily close prices via api.history(n) where needed
- Risk controls: avoid over-trading; use api.position() to check current holdings
- Aim for readable logic with clear variable names
- Strategy idea:
{idea}

Example structure:
```python
class Strategy:
    def __init__(self):
        # Initialize strategy state variables only
        pass
    
    def on_bar(self, api):
        # Use api methods here to implement trading logic
        pass
```"""

## 11) Generators (frontier + open‑source)

In [31]:
def gen_frontier(provider: str, model: str, idea: str, temperature: float = 0.2, max_tokens: int = 1200) -> str:
    prompt = TEMPLATE_USER.format(idea=idea)
    if provider == "OpenAI":
        if openai_client is None:
            raise RuntimeError("OpenAI client not configured")
        try:
            resp = openai_client.responses.create(
                model=model,
                input=[
                    {"role": "system", "content": SYSTEM},
                    {"role": "user", "content": prompt},
                ],
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
            text = resp.output_text
        except Exception:
            chat = openai_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM},
                    {"role": "user", "content": prompt},
                ],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            text = chat.choices[0].message.content
    elif provider == "Anthropic":
        if anthropic_client is None:
            raise RuntimeError("Anthropic client not configured")
        msg = anthropic_client.messages.create(
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
            system=SYSTEM,
            messages=[{"role": "user", "content": prompt}],
        )
        parts = []
        for blk in msg.content:
            if getattr(blk, "type", "") == "text":
                parts.append(blk.text)
        text = "\n".join(parts)
    else:
        raise ValueError("Unknown provider")
    
    # Clean up the generated code
    text = text.strip()
    # Remove markdown code blocks if present
    text = re.sub(r"```python\n?", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\n?```", "", text)
    # Remove any example usage or comments at the end
    lines = text.split('\n')
    clean_lines = []
    in_class = False
    for line in lines:
        if line.strip().startswith('class Strategy'):
            in_class = True
        elif in_class and line.strip() and not line.startswith(' ') and not line.startswith('\t'):
            # End of class definition
            break
        if in_class or line.strip().startswith('import ') or line.strip().startswith('from '):
            clean_lines.append(line)
    
    return '\n'.join(clean_lines).strip()

def gen_open_source(model_id: str, idea: str, temperature: float = 0.2, top_p: float = 0.95, max_new_tokens: int = 1200) -> str:
    tok, mdl = load_os_model(model_id)
    prompt = TEMPLATE_USER.format(idea=idea)
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": prompt},
    ]
    
    # Prepare input with better error handling
    try:
        input_ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    except Exception as e:
        print(f"⚠️ Chat template failed: {e}")
        # Fallback to simple concatenation
        full_prompt = f"{SYSTEM}\n\n{prompt}"
        input_ids = tok.encode(full_prompt, return_tensors="pt")
    
    # Handle device placement more carefully
    model_device = next(mdl.parameters()).device if hasattr(mdl, 'parameters') else torch.device('cpu')
    input_ids = input_ids.to(model_device)
    
    print(f"🔄 Generating with model on {model_device}, input shape: {input_ids.shape}")
    
    with torch.no_grad():
        try:
            out = mdl.generate(
                input_ids=input_ids,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                do_sample=True,
                pad_token_id=tok.eos_token_id,
            )
        except Exception as e:
            print(f"⚠️ Generation failed with advanced settings: {e}")
            # Fallback with simpler settings
            out = mdl.generate(
                input_ids=input_ids,
                max_new_tokens=min(max_new_tokens, 512),
                do_sample=False,  # Greedy decoding
                pad_token_id=tok.eos_token_id,
            )
    
    gen = out[0, input_ids.shape[1]:]
    text = tok.decode(gen, skip_special_tokens=True).strip()
    
    # Clean up the generated code
    # Remove markdown code blocks if present
    text = re.sub(r"```python\n?", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\n?```", "", text)
    # Remove any example usage or comments at the end
    lines = text.split('\n')
    clean_lines = []
    in_class = False
    for line in lines:
        if line.strip().startswith('class Strategy'):
            in_class = True
        elif in_class and line.strip() and not line.startswith(' ') and not line.startswith('\t'):
            # End of class definition
            break
        if in_class or line.strip().startswith('import ') or line.strip().startswith('from '):
            clean_lines.append(line)
    
    return '\n'.join(clean_lines).strip()

## 12) Safe exec of generated code

We execute the LLM output in a restricted global scope to extract `Strategy`.

In [32]:
import builtins as _builtins

SAFE_BUILTINS = {
    "abs": abs, "min": min, "max": max, "sum": sum, "len": len, "range": range, "enumerate": enumerate, "round": round,
    "float": float, "int": int, "bool": bool, "list": list, "dict": dict, "set": set, "tuple": tuple, "zip": zip,
    "print": print, "str": str, "isinstance": isinstance, "hasattr": hasattr, "getattr": getattr, "setattr": setattr,
    "__import__": __import__  # Add __import__ to allow import statements
}

def load_strategy_from_code(code: str):
    # basic sanitation: no imports other than allowed stdlib
    forbidden_imports = []
    import_matches = re.findall(r"\b(?:import|from)\s+([a-zA-Z0-9_\.]+)", code)
    
    for name in import_matches:
        base_module = name.split('.')[0]
        if base_module not in {"math", "statistics", "random"}:
            forbidden_imports.append(name)
    
    if forbidden_imports:
        raise ValueError(f"Only stdlib imports 'math', 'statistics', 'random' are allowed; found: {forbidden_imports}")
    
    # Create safe execution environment
    safe_globals = {
        "__builtins__": SAFE_BUILTINS, 
        "math": __import__("math"), 
        "statistics": __import__("statistics"), 
        "random": __import__("random")
    }
    local_scope = {}
    
    try:
        exec(code, safe_globals, local_scope)
    except Exception as e:
        raise ValueError(f"Code execution failed: {e}")
    
    # Look for Strategy class
    Strategy = local_scope.get("Strategy") or safe_globals.get("Strategy")
    if Strategy is None:
        raise ValueError("Strategy class not found in generated code")
    
    try:
        return Strategy()
    except Exception as e:
        raise ValueError(f"Strategy instantiation failed: {e}")

## 13) Gradio App

In [33]:
DEFAULT_IDEA = """
Moving-average crossover: buy when SMA(10) crosses above SMA(30); sell to flat when SMA(10) crosses below SMA(30).
Use order_target_percent(1.0) to be fully invested on buy signal; order_target_percent(0.0) to exit on sell.
Include warmup checks (history length) to avoid errors.
"""

def do_generate(mode, frontier_model, os_model, idea, temperature, top_p):
    try:
        if mode == "Open‑Source (HF Transformers)":
            code = gen_open_source(os_model, idea, temperature=temperature, top_p=top_p)
        elif mode == "Frontier (OpenAI)":
            code = gen_frontier("OpenAI", frontier_model, idea, temperature=temperature)
        else:
            code = gen_frontier("Anthropic", frontier_model, idea, temperature=temperature)
        return code
    except Exception as e:
        return f"⚠️ Generation error: {e}"

def do_backtest(code, symbol, start, end, interval, initial_cash, fee_bps, slippage_bps):
    if not code or len(code.strip()) == 0:
        return "⚠️ Generate a strategy first.", None, None, None, None
    
    if code.startswith("⚠️"):
        return code, None, None, None, None
    
    try:
        strat = load_strategy_from_code(code)
    except Exception as e:
        return f"⚠️ Strategy load error: {e}\n\nGenerated code:\n{code}", None, None, None, None

    try:
        df = load_ohlcv(symbol, start, end, interval=interval)
    except Exception as e:
        return f"⚠️ Data load error: {e}", None, None, None, None

    try:
        result = backtest(df, symbol, strat, initial_cash=initial_cash, fee_bps=fee_bps, slippage_bps=slippage_bps)
    except Exception as e:
        return f"⚠️ Backtest error: {e}", None, None, None, None

    try:
        # Plot equity curve
        fig1 = plt.figure()
        result["equity"].plot()
        plt.title(f"Equity Curve — {symbol}")
        plt.xlabel("Date"); plt.ylabel("Equity ($)")

        # Trades table preview
        trades = result["trades"]
        metrics = result["metrics"]
        metrics_json = json.dumps(metrics, indent=2)

        # Save artifacts
        tmp = tempfile.mkdtemp()
        code_path = os.path.join(tmp, "strategy.py")
        with open(code_path, "w", encoding="utf-8") as f:
            f.write(code)
        eq_csv = os.path.join(tmp, "equity.csv")
        result["equity"].to_csv(eq_csv, header=True)
        tr_csv = os.path.join(tmp, "trades.csv")
        trades.to_csv(tr_csv, index=False)

        return metrics_json, fig1, code_path, eq_csv, tr_csv
    except Exception as e:
        return f"⚠️ Output processing error: {e}", None, None, None, None

with gr.Blocks(title="LLM Trading Code Generator & Simulator") as app:
    gr.Markdown("## LLM Trading Code Generator & Simulator (Lesson 110)")
    with gr.Row():
        with gr.Column(scale=1):
            mode = gr.Radio(
                label="Generation Path",
                value="Open‑Source (HF Transformers)",
                choices=["Open‑Source (HF Transformers)", "Frontier (OpenAI)", "Frontier (Anthropic)"]
            )
            frontier_model = gr.Textbox(label="Frontier model", value="gpt-4o-mini", placeholder="e.g., gpt-4o-mini / claude-3-5-sonnet-20240620")
            os_model = gr.Dropdown(label="Open‑source model", value=OS_DEFAULT, choices=OS_CHOICES)
            temperature = gr.Slider(0.0, 1.5, value=0.2, step=0.05, label="Temperature (OS only)")
            top_p = gr.Slider(0.1, 1.0, value=0.95, step=0.05, label="top_p (OS only)")
            idea = gr.Textbox(label="Strategy idea", value=DEFAULT_IDEA, lines=6)
            btn_gen = gr.Button("1) Generate Strategy", variant="primary")
            code = gr.Code(label="Generated Strategy")
        with gr.Column(scale=1):
            symbol = gr.Textbox(label="Symbol", value="AAPL")
            start = gr.Textbox(label="Start (YYYY-MM-DD)", value="2019-01-01")
            end = gr.Textbox(label="End (YYYY-MM-DD)", value="2024-12-31")
            interval = gr.Dropdown(label="Interval", value="1d", choices=["1d","1wk","1mo"])
            initial_cash = gr.Number(label="Initial cash ($)", value=100000)
            fee_bps = gr.Number(label="Fee (bps)", value=1.0)
            slippage_bps = gr.Number(label="Slippage (bps)", value=1.0)
            btn_bt = gr.Button("2) Backtest", variant="primary")
        with gr.Column(scale=1):
            metrics = gr.Textbox(label="Metrics (JSON)", lines=12)
            fig1 = gr.Plot(label="Equity Curve")
            out_code = gr.File(label="strategy.py")
            out_eq = gr.File(label="equity.csv")
            out_tr = gr.File(label="trades.csv")

    btn_gen.click(fn=do_generate, inputs=[mode, frontier_model, os_model, idea, temperature, top_p], outputs=[code])
    btn_bt.click(fn=do_backtest, inputs=[code, symbol, start, end, interval, initial_cash, fee_bps, slippage_bps], outputs=[metrics, fig1, out_code, out_eq, out_tr])

print("✅ UI ready. In Colab/Jupyter, run: app.launch(share=True)")

✅ UI ready. In Colab/Jupyter, run: app.launch(share=True)


### 14) Launch the app

In [34]:
# Uncomment to launch inside the notebook
app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://a1a45f4d9745ae04ba.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Notes
- **Free data API**: This notebook uses `yfinance`, an open-source Python wrapper for Yahoo Finance's publicly available endpoints (no API key). It is widely used in research/education. Availability is not guaranteed; see the library docs and Yahoo terms for details.
- **GPU**: The open‑source path uses 4‑bit quantization and runs well on a **T4‑16GB** GPU. If you hit OOM, switch to `microsoft/Phi-3-mini-4k-instruct` or lower `max_new_tokens` in the generator.
- **Safety**: The generated code runs with limited builtins and no external imports (except `math`, `statistics`, `random`). Still, review outputs before running.
- **Financial warning**: This is a **toy simulator**. Results are hypothetical and for educational use only. Do **not** deploy any generated code to real markets.